# ◈ ARCANA — Stage 2: Feature Engineering
**Regime-aware statistical arbitrage platform**

This notebook computes all features required for downstream pipeline stages:
- `features_market.csv` → feeds the HMM Regime Engine (Stage 3)
- `features_returns.csv` → feeds Stat Arb + RV Engines (Stage 4)
- `features_volatility.csv` → feeds Portfolio Construction (Stage 6)
- `features_beta.csv` → feeds Factor Neutralisation (Stage 4A)
- `correlation_latest.csv` → feeds Pair Clustering (Stage 4A)

---

## 0 · Imports & Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Arcana plot style ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor'  : '#0d0f14',
    'axes.facecolor'    : '#0d0f14',
    'axes.edgecolor'    : '#2a2d35',
    'axes.labelcolor'   : '#8b8fa8',
    'axes.titlecolor'   : '#e2e4ed',
    'axes.titlesize'    : 13,
    'axes.labelsize'    : 11,
    'axes.grid'         : True,
    'grid.color'        : '#1e2029',
    'grid.linewidth'    : 0.6,
    'xtick.color'       : '#555870',
    'ytick.color'       : '#555870',
    'xtick.labelsize'   : 9,
    'ytick.labelsize'   : 9,
    'text.color'        : '#e2e4ed',
    'legend.facecolor'  : '#12141a',
    'legend.edgecolor'  : '#2a2d35',
    'legend.fontsize'   : 9,
    'figure.titlesize'  : 15,
    'lines.linewidth'   : 1.4,
    'font.family'       : 'monospace',
})

# Arcana colour palette
C_GOLD    = '#c9a84c'
C_TEAL    = '#3ec9b0'
C_BLUE    = '#4f8ef7'
C_RED     = '#e05c6b'
C_PURPLE  = '#9b7fe8'
C_GREY    = '#555870'
C_WHITE   = '#e2e4ed'

# ── CONFIG ────────────────────────────────────────────────────────────────
DATA_DIR   = r'C:\Arbion Research\Stage 1 data layer\Universe_stock data'
OUTPUT_DIR = r'C:\Arbion Research\Stage 2 feature engineering'

ROLLING_VOL_SHORT = 10
ROLLING_VOL_LONG  = 20
ROLLING_BETA      = 60
ROLLING_CORR      = 60
START_DATE        = '2010-01-01'
END_DATE          = '2026-04-10'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✓ Configuration loaded')
print(f'  Data   → {DATA_DIR}')
print(f'  Output → {OUTPUT_DIR}')
print(f'  Period → {START_DATE}  to  {END_DATE}')

---
## Step 1 · Load All Stock CSVs

In [ ]:
csv_files = list(Path(DATA_DIR).glob('*.csv'))
print(f'Found {len(csv_files)} CSV files\n')

prices  = {}
skipped = []

for f in sorted(csv_files):
    ticker = f.stem
    try:
        df = pd.read_csv(f, parse_dates=['Date'])
        df = df.rename(columns={'Date': 'date'})
        df = df.set_index('date').sort_index()
        df = df.loc[START_DATE:END_DATE]
        if 'Close' not in df.columns:
            skipped.append((ticker, 'missing Close column'))
            continue
        if len(df) < 100:
            skipped.append((ticker, f'only {len(df)} rows'))
            continue
        prices[ticker] = df['Close']
    except Exception as e:
        skipped.append((ticker, str(e)))

print(f'✓ Loaded  : {len(prices)} tickers')
if skipped:
    print(f'✗ Skipped : {len(skipped)} tickers')
    for t, reason in skipped:
        print(f'    {t:10s}  →  {reason}')

In [ ]:
# ── Visualise: data coverage heatmap ──────────────────────────────────────
price_matrix_raw = pd.DataFrame(prices).sort_index()

# Build a boolean availability matrix — sample every 30 trading days to keep it readable
avail = price_matrix_raw.notna().astype(int)
avail_sample = avail.iloc[::30]

fig, ax = plt.subplots(figsize=(18, 6))
fig.suptitle('ARCANA  ·  Step 1 — Data Coverage Map', fontsize=14, color=C_GOLD, y=1.01)

# Sort tickers alphabetically for clean display
avail_T = avail_sample.T.sort_index()

im = ax.imshow(
    avail_T.values,
    aspect='auto',
    cmap=sns.color_palette(['#1a1d26', C_TEAL], as_cmap=True),
    interpolation='nearest'
)

# x-axis: dates
n_cols   = avail_sample.shape[0]
x_ticks  = np.linspace(0, n_cols - 1, 8, dtype=int)
x_labels = [str(avail_sample.index[i].year) for i in x_ticks]
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, color='#555870')

# y-axis: tickers
ax.set_yticks(range(len(avail_T)))
ax.set_yticklabels(avail_T.index, fontsize=7, color='#8b8fa8')

ax.set_xlabel('Year', color='#8b8fa8')
ax.set_ylabel('Ticker', color='#8b8fa8')
ax.set_title(f'{len(prices)} tickers  ·  {START_DATE} → {END_DATE}', color='#8b8fa8', fontsize=10)

# legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=C_TEAL,  label='Data present'),
    Patch(facecolor='#1a1d26', edgecolor='#2a2d35', label='Missing'),
]
ax.legend(handles=legend_elements, loc='lower right', framealpha=0.8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'viz_step1_coverage.png'), dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print(f'✓ Chart saved: viz_step1_coverage.png')

---
## Step 2 · Build Aligned Price Matrix

In [ ]:
price_matrix = pd.DataFrame(prices).sort_index()
price_matrix = price_matrix.ffill(limit=2)  # fill up to 2 days (holidays/halts)

print(f'Price matrix shape : {price_matrix.shape[0]} trading days  ×  {price_matrix.shape[1]} tickers')
print(f'Date range         : {price_matrix.index[0].date()}  →  {price_matrix.index[-1].date()}')

# Missing data summary
missing_pct = (price_matrix.isnull().sum() / len(price_matrix) * 100).sort_values(ascending=False)
bad = missing_pct[missing_pct > 5]
if len(bad):
    print(f'\nTickers with >5% missing after forward-fill:')
    for t, pct in bad.items():
        print(f'    {t:10s}  →  {pct:.1f}% missing')
else:
    print('\n✓ All tickers have <5% missing data after forward-fill')

In [ ]:
# ── Visualise: normalised price series (select key tickers) ───────────────
HIGHLIGHT = ['SPY', 'AAPL', 'MSFT', 'JPM', 'XOM', 'AMZN']
highlight  = [t for t in HIGHLIGHT if t in price_matrix.columns]

# Normalise to 100 at start
norm = (price_matrix[highlight] / price_matrix[highlight].iloc[0]) * 100

fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle('ARCANA  ·  Step 2 — Normalised Price Series (base = 100)', color=C_GOLD)

colors = [C_GOLD, C_TEAL, C_BLUE, C_RED, C_PURPLE, C_WHITE]
for i, col in enumerate(highlight):
    lw = 2.0 if col == 'SPY' else 1.2
    ax.plot(norm.index, norm[col], color=colors[i % len(colors)], label=col, linewidth=lw)

# Shade key events
events = [
    ('2020-02-19', '2020-03-23', 'COVID crash',     '#e05c6b'),
    ('2022-01-03', '2022-10-13', '2022 bear market', '#c9a84c'),
]
for s, e, label, col in events:
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), alpha=0.08, color=col)
    ax.text(pd.Timestamp(s), ax.get_ylim()[1] * 0.97, f' {label}',
            fontsize=7, color=col, va='top')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_xlabel('Date')
ax.set_ylabel('Normalised price (base = 100)')
ax.legend(loc='upper left', ncol=len(highlight))
ax.set_xlim(price_matrix.index[0], price_matrix.index[-1])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'viz_step2_prices.png'), dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('✓ Chart saved: viz_step2_prices.png')

---
## Step 3 · Daily Returns

In [ ]:
returns = price_matrix.pct_change().iloc[1:]
returns.to_csv(os.path.join(OUTPUT_DIR, 'features_returns.csv'))
print(f'✓ features_returns.csv  —  shape {returns.shape}')
print(f'  Mean daily return (SPY) : {returns["SPY"].mean()*100:.4f}%')
print(f'  Std  daily return (SPY) : {returns["SPY"].std()*100:.4f}%')
print(f'  Min  daily return (SPY) : {returns["SPY"].min()*100:.2f}%  ({returns["SPY"].idxmin().date()})')
print(f'  Max  daily return (SPY) : {returns["SPY"].max()*100:.2f}%  ({returns["SPY"].idxmax().date()})')

In [ ]:
# ── Visualise: SPY returns + rolling vol + return distribution ────────────
spy = returns['SPY']
spy_vol_20 = spy.rolling(20).std() * np.sqrt(252)

fig, axes = plt.subplots(3, 1, figsize=(14, 10),
                         gridspec_kw={'height_ratios': [2, 1.5, 1.5]})
fig.suptitle('ARCANA  ·  Step 3 — Daily Returns Analysis (SPY)', color=C_GOLD, fontsize=14)

# Panel 1: daily returns bar
ax = axes[0]
colors_bar = np.where(spy >= 0, C_TEAL, C_RED)
ax.bar(spy.index, spy * 100, color=colors_bar, width=1.0, alpha=0.7)
ax.set_ylabel('Daily return (%)')
ax.set_title('SPY daily returns', color=C_WHITE, fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.axhline(0, color=C_GREY, linewidth=0.5)

# Panel 2: rolling realised volatility
ax2 = axes[1]
ax2.fill_between(spy_vol_20.index, spy_vol_20 * 100, alpha=0.4, color=C_GOLD)
ax2.plot(spy_vol_20.index, spy_vol_20 * 100, color=C_GOLD, linewidth=1.0)
ax2.set_ylabel('Annualised vol (%)')
ax2.set_title('20-day rolling realised volatility', color=C_WHITE, fontsize=11)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Shade high-vol regimes (vol > 30%)
ax2.axhline(30, color=C_RED, linewidth=0.8, linestyle='--', alpha=0.6)
ax2.text(spy_vol_20.index[-1], 31, ' crisis threshold', color=C_RED, fontsize=8, va='bottom')

# Panel 3: return histogram with normal overlay
ax3 = axes[2]
spy_pct = spy * 100
ax3.hist(spy_pct, bins=120, color=C_BLUE, alpha=0.6, density=True, label='Empirical')

from scipy.stats import norm as sp_norm
x_range = np.linspace(spy_pct.min(), spy_pct.max(), 300)
ax3.plot(x_range, sp_norm.pdf(x_range, spy_pct.mean(), spy_pct.std()),
         color=C_GOLD, linewidth=1.5, label='Normal fit')
ax3.set_xlabel('Daily return (%)')
ax3.set_ylabel('Density')
ax3.set_title('Return distribution  —  fat tails visible (key: use Student-t HMM)', color=C_WHITE, fontsize=11)
ax3.legend()
ax3.set_xlim(-8, 8)

# Kurtosis annotation
kurt = spy_pct.kurtosis()
ax3.text(0.02, 0.92, f'Excess kurtosis: {kurt:.2f}  (normal = 0)',
         transform=ax3.transAxes, color=C_RED, fontsize=9)

for ax in axes:
    ax.set_xlim(returns.index[0], returns.index[-1])
axes[2].set_xlim(-8, 8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'viz_step3_returns.png'), dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('✓ Chart saved: viz_step3_returns.png')

---
## Step 4 · Market-Level Features (HMM Inputs)

In [ ]:
if 'SPY' not in returns.columns:
    raise ValueError('SPY not found — required for market features')

spy_ret = returns['SPY']
stock_cols = [c for c in returns.columns if c != 'SPY']

market = pd.DataFrame(index=returns.index)
market['spy_return']      = spy_ret
market['spy_vol_10d']     = spy_ret.rolling(ROLLING_VOL_SHORT).std() * np.sqrt(252)
market['spy_vol_20d']     = spy_ret.rolling(ROLLING_VOL_LONG).std()  * np.sqrt(252)
market['spy_vol_20d_raw'] = spy_ret.rolling(ROLLING_VOL_LONG).std()
market['xs_dispersion']   = returns[stock_cols].std(axis=1)
market = market.dropna()

market.to_csv(os.path.join(OUTPUT_DIR, 'features_market.csv'))
print(f'✓ features_market.csv  —  shape {market.shape}')
print(f'\nFeature summary:')
print(market.describe().round(5).to_string())

In [ ]:
# ── Visualise: all 4 HMM input features ──────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle('ARCANA  ·  Step 4 — Market Features  (HMM Inputs)', color=C_GOLD, fontsize=14)

series = [
    ('spy_return',     'SPY daily return',             C_TEAL,   True),
    ('spy_vol_10d',    'SPY vol 10d annualised',        C_GOLD,   False),
    ('spy_vol_20d',    'SPY vol 20d annualised',        C_BLUE,   False),
    ('xs_dispersion',  'Cross-sectional dispersion',   C_PURPLE, False),
]

for i, (col, title, color, is_return) in enumerate(series):
    ax = axes[i]
    s = market[col] * 100 if col in ['spy_return', 'spy_vol_10d', 'spy_vol_20d'] else market[col] * 100
    if is_return:
        c_bar = np.where(market[col] >= 0, C_TEAL, C_RED)
        ax.bar(market.index, market[col] * 100, color=c_bar, width=1.0, alpha=0.7)
        ax.axhline(0, color=C_GREY, linewidth=0.4)
    else:
        ax.fill_between(market.index, market[col] * 100, alpha=0.25, color=color)
        ax.plot(market.index, market[col] * 100, color=color, linewidth=1.0)
    ax.set_ylabel('%', fontsize=9)
    ax.set_title(title, color=C_WHITE, fontsize=10)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].set_xlim(market.index[0], market.index[-1])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'viz_step4_market_features.png'), dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('✓ Chart saved: viz_step4_market_features.png')

---
## Step 5 · Rolling Volatility (All Assets)

In [ ]:
vol_matrix = returns.rolling(ROLLING_VOL_LONG).std() * np.sqrt(252)
vol_matrix.to_csv(os.path.join(OUTPUT_DIR, 'features_volatility.csv'))
print(f'✓ features_volatility.csv  —  shape {vol_matrix.shape}')

# Summary stats
latest_vol = vol_matrix.iloc[-1].sort_values(ascending=False)
print(f'\nCurrent (latest) annualised vol — top 10 most volatile:')
for t, v in latest_vol.head(10).items():
    print(f'    {t:10s}  {v*100:.1f}%')
print(f'\nCurrent (latest) annualised vol — top 10 least volatile:')
for t, v in latest_vol.tail(10).items():
    print(f'    {t:10s}  {v*100:.1f}%')

In [ ]:
# ── Visualise: vol distribution across universe at multiple snapshots ─────
snapshots = ['2012-01-03', '2016-01-04', '2020-03-20', '2022-06-15', str(vol_matrix.dropna().index[-1].date())]
snapshots = [s for s in snapshots if s in [str(d.date()) for d in vol_matrix.index]]
# Use nearest available date
snap_dates = []
for s in ['2012-01-03', '2016-01-04', '2020-03-20', '2022-06-15']:
    idx = vol_matrix.index.searchsorted(pd.Timestamp(s))
    snap_dates.append(vol_matrix.index[min(idx, len(vol_matrix.index)-1)])
snap_dates.append(vol_matrix.index[-1])

snap_colors = [C_BLUE, C_TEAL, C_RED, C_GOLD, C_PURPLE]
snap_labels = ['2012 (calm)', '2016 (moderate)', '2020 (COVID crash)', '2022 (bear market)', 'Latest']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ARCANA  ·  Step 5 — Asset Volatility Distribution', color=C_GOLD, fontsize=14)

# Left: KDE of vol distribution at each snapshot
ax = axes[0]
for i, (date, color, label) in enumerate(zip(snap_dates, snap_colors, snap_labels)):
    snap_vol = vol_matrix.loc[date].dropna() * 100
    snap_vol = snap_vol[snap_vol < 150]  # remove outliers
    snap_vol.plot.kde(ax=ax, color=color, label=f'{label} (med={snap_vol.median():.0f}%)', linewidth=1.4)

ax.set_xlabel('Annualised volatility (%)')
ax.set_ylabel('Density')
ax.set_title('Vol distribution across universe', color=C_WHITE, fontsize=10)
ax.set_xlim(0, 120)
ax.legend(fontsize=8)

# Right: vol fan chart — median and percentiles over time
ax2 = axes[1]
vol_pct = vol_matrix * 100
med  = vol_pct.median(axis=1)
p25  = vol_pct.quantile(0.25, axis=1)
p75  = vol_pct.quantile(0.75, axis=1)
p10  = vol_pct.quantile(0.10, axis=1)
p90  = vol_pct.quantile(0.90, axis=1)

ax2.fill_between(vol_pct.index, p10, p90, alpha=0.15, color=C_GOLD, label='10th-90th pct')
ax2.fill_between(vol_pct.index, p25, p75, alpha=0.25, color=C_GOLD, label='25th-75th pct')
ax2.plot(med.index, med, color=C_GOLD, linewidth=1.5, label='Median vol')

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.set_xlabel('Date')
ax2.set_ylabel('Annualised vol (%)')
ax2.set_title('Cross-universe vol fan chart over time', color=C_WHITE, fontsize=10)
ax2.legend(fontsize=8)
ax2.set_xlim(vol_pct.index[0], vol_pct.index[-1])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'viz_step5_volatility.png'), dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('✓ Chart saved: viz_step5_volatility.png')

---
## Step 6 · Rolling Beta vs SPY

In [ ]:
spy_ret_s = returns['SPY']
spy_var   = spy_ret_s.rolling(ROLLING_BETA).var()
beta_matrix = pd.DataFrame(index=returns.index, columns=stock_cols, dtype=float)

for ticker in stock_cols:
    cov = returns[ticker].rolling(ROLLING_BETA).cov(spy_ret_s)
    beta_matrix[ticker] = cov / spy_var

beta_matrix.to_csv(os.path.join(OUTPUT_DIR, 'features_beta.csv'))
print(f'✓ features_beta.csv  —  shape {beta_matrix.shape}')

# Latest betas
latest_beta = beta_matrix.iloc[-1].sort_values(ascending=False)
print(f'\nCurrent betas — top 10 highest:')
for t, b in latest_beta.head(10).items():
    print(f'    {t:10s}  β = {b:.3f}')
print(f'\nCurrent betas — top 10 lowest:')
for t, b in latest_beta.tail(10).items():
    print(f'    {t:10s}  β = {b:.3f}')

In [ ]:
# ── Visualise: beta over time for selected stocks + beta distribution ─────
BETA_WATCH = ['AAPL', 'AMZN', 'JPM', 'XOM', 'KO', 'TSLA']
beta_watch = [t for t in BETA_WATCH if t in beta_matrix.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ARCANA  ·  Step 6 — Rolling Beta vs SPY (60d window)', color=C_GOLD, fontsize=14)

# Left: time series of selected betas
ax = axes[0]
colors_list = [C_TEAL, C_BLUE, C_GOLD, C_RED, C_PURPLE, C_WHITE]
for i, ticker in enumerate(beta_watch):
    ax.plot(beta_matrix.index, beta_matrix[ticker],
            color=colors_list[i], label=ticker, linewidth=1.2, alpha=0.85)

ax.axhline(1.0, color=C_GREY, linewidth=0.7, linestyle='--', label='β = 1')
ax.axhline(0.0, color=C_GREY, linewidth=0.4, linestyle=':')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylabel('Beta')
ax.set_title('Rolling 60d beta — selected tickers', color=C_WHITE, fontsize=10)
ax.legend(ncol=2, fontsize=8)
ax.set_xlim(beta_matrix.index[0], beta_matrix.index[-1])
ax.set_ylim(-1, 3.5)

# Right: current beta distribution histogram
ax2 = axes[1]
current_betas = beta_matrix.iloc[-1].dropna()
current_betas = current_betas[current_betas.abs() < 5]  # remove outliers

ax2.hist(current_betas, bins=40, color=C_BLUE, alpha=0.65, edgecolor='#0d0f14', linewidth=0.3)
ax2.axvline(1.0, color=C_GOLD, linewidth=1.5, linestyle='--', label=f'β = 1')
ax2.axvline(current_betas.mean(), color=C_TEAL, linewidth=1.5, linestyle='-',
            label=f'Mean β = {current_betas.mean():.2f}')
ax2.set_xlabel('Beta')
ax2.set_ylabel('Count')
ax2.set_title('Current beta distribution (all tickers)', color=C_WHITE, fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'viz_step6_beta.png'), dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('✓ Chart saved: viz_step6_beta.png')

---
## Step 7 · Correlation Matrix (Latest 60d)

In [ ]:
last_idx    = returns.index[-1]
start_corr  = returns.index[max(0, len(returns.index) - ROLLING_CORR)]
corr_window = returns.loc[start_corr:last_idx, stock_cols]
corr_matrix = corr_window.corr()

corr_matrix.to_csv(os.path.join(OUTPUT_DIR, 'correlation_latest.csv'))
print(f'✓ correlation_latest.csv  —  {corr_matrix.shape[0]}×{corr_matrix.shape[1]} matrix')
print(f'  Window: {start_corr.date()}  →  {last_idx.date()}')

# Flatten upper triangle for stats
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
flat  = upper.stack()
print(f'\nPairwise correlation stats (upper triangle):')
print(f'  Mean   : {flat.mean():.4f}')
print(f'  Median : {flat.median():.4f}')
print(f'  Std    : {flat.std():.4f}')
print(f'  Min    : {flat.min():.4f}')
print(f'  Max    : {flat.max():.4f}')

In [ ]:
# ── Visualise: correlation heatmap + distribution ─────────────────────────
# Sort tickers by sector (approximate by alphabetical — user can reorder)
sorted_tickers = sorted(corr_matrix.columns)
corr_sorted = corr_matrix.loc[sorted_tickers, sorted_tickers]

fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                         gridspec_kw={'width_ratios': [3, 1]})
fig.suptitle('ARCANA  ·  Step 7 — Correlation Matrix (latest 60d)', color=C_GOLD, fontsize=14)

# Left: heatmap
ax = axes[0]
cmap = sns.diverging_palette(220, 20, s=80, l=40, as_cmap=True)
mask = np.triu(np.ones_like(corr_sorted, dtype=bool), k=1)  # show full matrix

sns.heatmap(
    corr_sorted,
    ax=ax,
    cmap=cmap,
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.0,
    xticklabels=True,
    yticklabels=True,
    cbar_kws={'shrink': 0.6, 'label': 'Correlation'}
)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=5.5, rotation=90, color='#8b8fa8')
ax.set_yticklabels(ax.get_yticklabels(), fontsize=5.5, rotation=0, color='#8b8fa8')
ax.set_title(f'All {len(sorted_tickers)} tickers  ·  {start_corr.date()} → {last_idx.date()}',
             color=C_WHITE, fontsize=10)

# Right: distribution of pairwise correlations
ax2 = axes[1]
ax2.hist(flat, bins=60, color=C_PURPLE, alpha=0.7, orientation='horizontal',
         edgecolor='#0d0f14', linewidth=0.2)
ax2.axhline(flat.mean(), color=C_GOLD, linewidth=1.5, linestyle='--',
            label=f'Mean = {flat.mean():.2f}')
ax2.axhline(0, color=C_GREY, linewidth=0.6, linestyle=':')
ax2.set_xlabel('Count')
ax2.set_ylabel('Pairwise correlation')
ax2.set_title('Correlation\ndistribution', color=C_WHITE, fontsize=10)
ax2.legend(fontsize=8)
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'viz_step7_correlation.png'), dpi=150, bbox_inches='tight', facecolor='#0d0f14')
plt.show()
print('✓ Chart saved: viz_step7_correlation.png')

---
## Stage 2 — Summary

In [ ]:
output_files = [
    ('features_market.csv',      market.shape,       'HMM regime engine (Stage 3)'),
    ('features_returns.csv',     returns.shape,      'Stat arb + RV (Stage 4)'),
    ('features_volatility.csv',  vol_matrix.shape,   'Portfolio construction (Stage 6)'),
    ('features_beta.csv',        beta_matrix.shape,  'Factor neutralisation (Stage 4A)'),
    ('correlation_latest.csv',   corr_matrix.shape,  'Pair clustering (Stage 4A)'),
]

print('=' * 65)
print('  ARCANA — Stage 2 Complete')
print('=' * 65)
print(f'  Output directory: {OUTPUT_DIR}\n')
print(f'  {"File":<35} {"Shape":<18} {"Next stage"}')
print(f'  {"-"*35} {"-"*18} {"-"*25}')
for fname, shape, dest in output_files:
    print(f'  {fname:<35} {str(shape):<18} → {dest}')

print(f'\n  Visualisations saved:')
for i in range(1, 8):
    fname = f'viz_step{i}_*.png'
    print(f'    viz_step{i}_*.png')

print(f'\n  Next: Stage 3 — HMM Regime Engine')
print(f'  Input: features_market.csv')
print('=' * 65)